## Function: Setup PostGIS + OSM Workflow 🗄️

In this notebook, you'll learn how to build the `setup_postgis_osm()` function step by step. This workflow sets up a spatial database, downloads OpenStreetMap data, and loads it into PostGIS.

### 🎯 What This Function Does
- Connect to a PostgreSQL server
- Create a new database
- Enable PostGIS extension
- Download OSM data from Geofabrik
- Load OSM data into PostGIS

### 🔧 Function Signature
```python
def setup_postgis_osm(...):
```

### 📍 Where This Function Goes:
**Target File**: `src/postgis_setup.py`
**Function Name**: `setup_postgis_osm()`

---

### ⚙️ Step 0: Select the Correct Python Kernel

Make sure the notebook is using **python-gis-development (.venv)** before running cells.

This ensures all required libraries and system tools are available.

### 📚 Step 1: Import Required Libraries

We will use the following tools:
- `psycopg2`: connect to PostgreSQL
- `requests`: download OSM data
- `subprocess`: run system commands (`osm2pgsql`)
- `os`: manage file paths

**💡 These will be used in our function!**

In [ ]:
import os
import requests
import subprocess
import psycopg2

### 🔌 Step 2: Connect to PostgreSQL Server

We first connect to the default `postgres` database.

This allows us to create a new database.

**💡 This will be used in our function!**

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

conn.autocommit = True
cur = conn.cursor()

print("Connected to PostgreSQL server")

### 🗄️ Step 3: Create a New Database

We create a new database where OSM data will be stored.

**💡 This will be used in our function!**

In [ ]:
cur.execute("CREATE DATABASE osm_db;")

cur.close()
conn.close()

print("Database created")

### 🌍 Step 4: Enable PostGIS Extension

PostGIS adds spatial capabilities to PostgreSQL.

Without this step, we cannot store or analyze spatial data.

**💡 This will be used in our function!**

In [ ]:
conn = psycopg2.connect(
    dbname="osm_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE EXTENSION postgis;")

cur.close()
conn.close()

print("PostGIS enabled")

### ⬇️ Step 5: Download OSM Data from Geofabrik

Geofabrik provides pre-packaged OpenStreetMap extracts.

We will download Hawaii data as our example.

**💡 This will be used in our function!**

In [ ]:
osm_url = "https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"

data_path = "data/osm"
os.makedirs(data_path, exist_ok=True)

file_path = os.path.join(data_path, osm_url.split("/")[-1])

if not os.path.exists(file_path):
    print("Downloading OSM data...")
    r = requests.get(osm_url, stream=True)
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(1024):
            f.write(chunk)

print("Download complete")

### 📥 Step 6: Load OSM Data into PostGIS

We use `osm2pgsql` to convert OSM `.pbf` data into database tables.

**Command Explanation:**
- `osm2pgsql`: converts OSM data to SQL
- `-d`: database name
- `-U`: database user
- `--create`: create new tables
- `--slim`: required for larger datasets

**💡 This will be used in our function!**

In [ ]:
cmd = [
    "osm2pgsql",
    "-d", "osm_db",
    "-U", "postgres",
    "--create",
    "--slim",
    file_path
]

subprocess.run(cmd, check=True)

print("OSM data loaded into PostGIS")

### 🧩 Step 7: Build the Complete Function

Now we combine all steps into a reusable function.

In [ ]:
def setup_postgis_osm(db_name="osm_db", osm_url=None):
    import os, requests, subprocess, psycopg2

    data_path = "data/osm"
    os.makedirs(data_path, exist_ok=True)

    file_path = os.path.join(data_path, osm_url.split("/")[-1])

    if not os.path.exists(file_path):
        r = requests.get(osm_url, stream=True)
        with open(file_path, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

    conn = psycopg2.connect(dbname="postgres", user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE {db_name};")
    cur.close()
    conn.close()

    conn = psycopg2.connect(dbname=db_name, user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION postgis;")
    cur.close()
    conn.close()

    subprocess.run([
        "osm2pgsql",
        "-d", db_name,
        "-U", "postgres",
        "--create",
        "--slim",
        file_path
    ], check=True)

    return True

### 🧪 Step 8: Test the Function

In [ ]:
setup_postgis_osm(
    osm_url="https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
)

print("Setup complete")